In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:


from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType
)

file_path = "/Workspace/Users/vadiveldeepika07@gmail.com/MALL_PROJECT/Mall_Customers.csv"

schema = StructType([
    StructField("CustomerID", IntegerType()),
    StructField("Gender", StringType()),
    StructField("Age", IntegerType()),
    StructField("Annual Income (k$)", IntegerType()),
    StructField("Spending Score (1-100)", IntegerType())
])

df = spark.read.csv(
    file_path,
    header=True,
    schema=schema
)

# Standardize column names
df = (df
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumnRenamed("Gender", "gender")
    .withColumnRenamed("Age", "age")
    .withColumnRenamed("Annual Income (k$)", "annual_income_k")
    .withColumnRenamed("Spending Score (1-100)", "spending_score")
)

# Display schema
df.printSchema()

# Display records
display(df.limit(10))

root
 |-- customer_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- annual_income_k: integer (nullable = true)
 |-- spending_score: integer (nullable = true)



customer_id,gender,age,annual_income_k,spending_score
1,Male,19,15,39
2,Male,21,15,81
3,Female,20,16,6
4,Female,23,16,77
5,Female,31,17,40
6,Female,22,17,76
7,Female,35,18,6
8,Female,23,18,94
9,Male,64,19,3
10,Female,30,19,72


##data quality checks

In [0]:
## row count and duplicates
print("Rows:", df.count())  
print("Duplicate IDs:", df.count() - df.dropDuplicates(["customer_id"]).count())  

Rows: 200
Duplicate IDs: 0


In [0]:
## null check
display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]))

customer_id,gender,age,annual_income_k,spending_score
0,0,0,0,0


In [0]:
## summary
display(df.describe())

summary,customer_id,gender,age,annual_income_k,spending_score
count,200,200,200,200,200
mean,100.5,null,38.85,60.56,50.2
stddev,57.879184513951124,null,13.969007331558883,26.264721165271254,25.823521668370162
min,1,Female,18,15,1
max,200,Male,70,137,99


In [0]:
## category check
display(df.groupBy("gender").count())

gender,count
Male,88
Female,112


In [0]:
from pyspark.sql.functions import col, median, coalesce, lit

In [0]:
## Calculate the median values
age_median = df.select(median("age")).first()[0]

income_median = df.select(median("annual_income_k")).first()[0]

score_median = df.select(median("spending_score")).first()[0]

In [0]:
## . Fill missing values
df_clean = (df
    .fillna({"gender": "Unknown"})
    .fillna({
        "age": age_median,
        "annual_income_k": income_median,
        "spending_score": score_median
    })
)

In [0]:
## Check whether missing values
from pyspark.sql.functions import sum, when

null_counts = df_clean.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_clean.columns
])

display(null_counts)

customer_id,gender,age,annual_income_k,spending_score
0,0,0,0,0


In [0]:
## Feature Engineering (Age Groups)
from pyspark.sql.functions import when, col

df = df.withColumn(
    "age_group",
    when((col("age") >= 0) & (col("age") <= 20), "Teenager")
    .when((col("age") > 20) & (col("age") <= 40), "Young Adult")
    .when((col("age") > 40) & (col("age") <= 60), "Middle Aged")
    .when((col("age") > 60) & (col("age") <= 100), "Senior")
    .otherwise("Unknown")
)

In [0]:
## save the cleaned data as a table
df.write.mode("overwrite").saveAsTable("workspace.default.mall_customers_clean")
